In [ ]:
BASE = '/kaggle/input/datasets/aneeshdighe/corals-classification/Bleached Corals and Healthy Corals Classification'

TRAIN_DIR = BASE + '/Training'
VAL_DIR   = BASE + '/Validation'
TEST_DIR  = BASE + '/Testing'

print("Train:", TRAIN_DIR)
print("Val:", VAL_DIR)
print("Test:", TEST_DIR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

IMG_SIZE = (224, 224)
BATCH    = 32

# Data Generators
train_gen = ImageDataGenerator(rescale=1./255, horizontal_flip=True, zoom_range=0.2, rotation_range=15)
val_gen   = ImageDataGenerator(rescale=1./255)
test_gen  = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH, class_mode='binary')
val_data   = val_gen.flow_from_directory(VAL_DIR,   target_size=IMG_SIZE, batch_size=BATCH, class_mode='binary')
test_data  = test_gen.flow_from_directory(TEST_DIR,  target_size=IMG_SIZE, batch_size=BATCH, class_mode='binary', shuffle=False)

print("Classes:", train_data.class_indices)

callbacks = [EarlyStopping(patience=5, restore_best_weights=True), ReduceLROnPlateau(patience=3, factor=0.5)]

# MODEL 1: Custom CNN
cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n=== Training Custom CNN ===")
cnn_history = cnn_model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)

# MODEL 2: VGG16
base = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False
vgg_model = models.Sequential([base, layers.GlobalAveragePooling2D(), layers.Dense(256, activation='relu'), layers.Dropout(0.5), layers.Dense(1, activation='sigmoid')])
vgg_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n=== Training VGG16 ===")
vgg_history = vgg_model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)

# EVALUATE BOTH
def evaluate_model(model, name, history):
    loss, acc = model.evaluate(test_data)
    print(f"\n{name} → Test Accuracy: {acc:.4f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title(f'{name} - Accuracy'); axes[0].legend()
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title(f'{name} - Loss'); axes[1].legend()
    plt.tight_layout(); plt.savefig(f'{name}_curves.png'); plt.show()
    preds = (model.predict(test_data) > 0.5).astype(int).flatten()
    labels = test_data.classes
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=list(test_data.class_indices.keys()), yticklabels=list(test_data.class_indices.keys()))
    plt.title(f'{name} - Confusion Matrix'); plt.savefig(f'{name}_cm.png'); plt.show()
    print(classification_report(labels, preds, target_names=list(test_data.class_indices.keys())))
    return acc

acc_cnn = evaluate_model(cnn_model, 'CustomCNN', cnn_history)
acc_vgg = evaluate_model(vgg_model, 'VGG16', vgg_history)

print("\n=== Model Comparison ===")
print(f"Custom CNN : {acc_cnn:.4f}")
print(f"VGG16      : {acc_vgg:.4f}")
print(f"Best model : {'VGG16' if acc_vgg > acc_cnn else 'Custom CNN'}")